# Improvement iteration 2: retrain the generator with hard distractors (GPU)

**Question.** Notebook 13 found that the gold-facts oracle mostly helps by removing distractor facts, and
Notebook 14 showed that simply passing fewer facts loses more than it gains. Does training the generator
with **hard distractors** (non-gold facts that share many words with the question, like the ones a
retriever ranks highly) make it more robust to the distractors it sees at prediction time?

**What changes from Notebook 11.** Only the training distractors: `hard_distractor_share = 0.5` in
`configs/generator_t5_base_hard.toml` fills half the distractor slots with the most question-similar
non-gold facts and the rest at random. Model, data, epochs, seed and decoding are the same. Hard
distractors are a lexical proxy: the Notebook 10 retriever's checkpoint was not kept, and its rankings on
its own training data would be in-sample anyway.

**Decision this iteration tests.** Whether to replace the Notebook 11 checkpoint in the pipeline. The
comparison is against the current pipeline (Notebook 11 checkpoint + grounded selection, dev 0.6195).

**Protocol.** Train on train; select the epoch by dev execution accuracy (top beam, as in Notebook 11);
report dev with top-beam and grounded selection. The test split is not loaded. Runs on a CUDA GPU
(Kaggle T4, about 2 h); `FINEVIDENCE_SMOKE_TEST=1` runs a tiny CPU check.

In [ ]:
from __future__ import annotations

import os

# Must be set before CUDA initializes; reduces fragmentation-related out-of-memory errors.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json
import platform
import random
import subprocess
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_linear_schedule_with_warmup

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.generation import (
    build_oracle_examples,
    build_retrieved_examples,
    build_training_examples,
    load_generator_config,
    load_rankings,
    score_program_prediction,
    select_grounded_candidate,
    summarize_scores,
    verify_rankings_file,
)

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
CONFIG_NAME = os.environ.get("FINEVIDENCE_GENERATOR_CONFIG", "generator_t5_base_hard.toml")
config = load_generator_config(PROJECT_ROOT / "configs" / CONFIG_NAME)

pretrained_name = (
    "hf-internal-testing/tiny-random-T5ForConditionalGeneration"
    if SMOKE_TEST
    else config.pretrained_name
)
epochs = 1 if SMOKE_TEST else config.epochs
num_beams = 1 if SMOKE_TEST else config.num_beams
suffix = "-smoke" if SMOKE_TEST else ""
model_dir = Path(str(config.model_dir) + suffix)
model_dir = model_dir if model_dir.is_absolute() else PROJECT_ROOT / model_dir
predictions_dir = PROJECT_ROOT / config.predictions_dir
rankings_path = config.rankings_path
rankings_path = rankings_path if rankings_path.is_absolute() else PROJECT_ROOT / rankings_path

random.seed(config.seed)
np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "Full training needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_amp = config.mixed_precision and device.type == "cuda"

commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "pretrained_name": pretrained_name,
    "epochs": epochs,
    "num_beams": num_beams,
    "batch_size": config.batch_size,
    "micro_batch_size": config.micro_batch_size,
    "eval_batch_size": config.eval_batch_size,
    "mixed_precision": use_amp,
    "seed": config.seed,
    "config": CONFIG_NAME,
    "hard_distractor_share": config.hard_distractor_share,
}
print(json.dumps(environment, indent=2))

## 1. Verified data, rankings, and generator inputs

The rankings file must match the SHA-256 recorded in the config, so it is the exact Notebook 10 output.
Training examples are rebuilt each epoch with a new distractor sample. Facts are rendered from the raw
context by key in every condition.

In [ ]:
raw_dir = PROJECT_ROOT / config.dataset_root
print(download_finqa_splits(raw_dir, (config.train_split, config.evaluation_split)))
verify_rankings_file(rankings_path, config.rankings_sha256)
rankings = load_rankings(rankings_path)
train = load_finqa_split(raw_dir / f"{config.train_split}.json")
dev = load_finqa_split(raw_dir / f"{config.evaluation_split}.json")
if SMOKE_TEST:
    train, dev = train[:40], dev[:20]

retrieved_examples = build_retrieved_examples(dev, rankings, facts_per_input=config.facts_per_input)
oracle_examples = build_oracle_examples(dev)
first_epoch_examples = build_training_examples(
    train,
    facts_per_input=config.facts_per_input,
    seed=config.seed,
    hard_distractor_share=config.hard_distractor_share,
)
print(
    {
        "train_records": len(train),
        "dev_records": len(dev),
        "test_loaded": False,
        "rankings_verified": str(rankings_path.name),
        "facts_per_input": config.facts_per_input,
        "hard_distractor_share": config.hard_distractor_share,
    }
)
for name, example in [
    ("train", first_epoch_examples[0]),
    ("dev retrieved", retrieved_examples[0]),
    ("dev oracle", oracle_examples[0]),
]:
    print(
        f"\n[{name}] {example.record_id}\n  source: {example.source[:400]}\n  target: {example.target}"
    )

## 2. Tokenized lengths

Sources longer than `max_source_length` are truncated from the end, which drops the lowest-ranked facts
first. Targets longer than `max_target_length` would be cut and could not be generated exactly.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_name)


def token_lengths(texts):
    return np.array([len(ids) for ids in tokenizer(list(texts)).input_ids])


length_rows = []
for name, texts, limit in [
    ("train source", [e.source for e in first_epoch_examples], config.max_source_length),
    ("train target", [e.target for e in first_epoch_examples], config.max_target_length),
    ("dev retrieved source", [e.source for e in retrieved_examples], config.max_source_length),
    ("dev oracle source", [e.source for e in oracle_examples], config.max_source_length),
]:
    lengths = token_lengths(texts)
    length_rows.append(
        {
            "input": name,
            "median": int(np.median(lengths)),
            "p95": int(np.percentile(lengths, 95)),
            "max": int(lengths.max()),
            "limit": limit,
            "over_limit": int((lengths > limit).sum()),
        }
    )
display(pd.DataFrame(length_rows))

## 3. Train the generator

AdamW with linear warmup and decay. Each optimizer step uses `batch_size` examples, processed as
`micro_batch_size` chunks whose losses are weighted by their share of target tokens, so the accumulated
gradient equals the full-batch gradient. The decoder cache is disabled during training. Each epoch draws a new distractor sample and a new example order, both
seeded from the config seed and the epoch. After every epoch the model decodes the retrieved dev inputs
with beam search, and the checkpoint with the best dev execution accuracy is kept. Decoding uses
`clean_up_tokenization_spaces=False`, because the default clean-up changes row labels such as `u.k .`.

In [ ]:
def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


model = AutoModelForSeq2SeqLM.from_pretrained(pretrained_name).to(device)


def encode_sources(examples):
    return tokenizer(
        [example.source for example in examples],
        max_length=config.max_source_length,
        truncation=True,
        padding=True,
        return_tensors="pt",
    ).to(device)


@torch.no_grad()
def generate(examples, label: str) -> list[str]:
    model.eval()
    predictions, started = [], time.time()
    batch_size = config.eval_batch_size
    report_every = max(1, len(examples) // (batch_size * 4)) * batch_size
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        with torch.autocast(device.type, enabled=use_amp):
            output = model.generate(
                **encode_sources(batch),
                max_new_tokens=config.max_target_length,
                num_beams=num_beams,
            )
        predictions.extend(
            tokenizer.batch_decode(
                output, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
        )
        done = min(start + batch_size, len(examples))
        if done % report_every == 0 or done == len(examples):
            print(
                f"  {label}: decoded {done}/{len(examples)} ({clock(time.time() - started)})",
                flush=True,
            )
    return predictions


def evaluate(examples, predictions):
    scores = [
        score_program_prediction(record, prediction, example.fact_texts)
        for record, example, prediction in zip(dev, examples, predictions, strict=True)
    ]
    return scores, summarize_scores(scores)


steps_per_epoch = -(-len(train) // config.batch_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(config.warmup_ratio * steps_per_epoch * epochs), steps_per_epoch * epochs
)
scaler = torch.amp.GradScaler(enabled=use_amp)
history, best_accuracy = [], -1.0
model_dir.mkdir(parents=True, exist_ok=True)
report_every = max(1, steps_per_epoch // 10)
run_started = time.time()
print(
    f"Training {epochs} epoch(s) x {steps_per_epoch} steps, batch {config.batch_size}, "
    f"beams {num_beams}, on {environment['device']}",
    flush=True,
)

for epoch in range(epochs):
    model.train()
    epoch_examples = build_training_examples(
        train,
        facts_per_input=config.facts_per_input,
        seed=config.seed + epoch,
        hard_distractor_share=config.hard_distractor_share,
    )
    random.Random(config.seed * 1_000_003 + epoch).shuffle(epoch_examples)
    started, losses = time.time(), []
    print(f"\nEpoch {epoch + 1}/{epochs}", flush=True)
    for step, start in enumerate(range(0, len(epoch_examples), config.batch_size), start=1):
        batch = epoch_examples[start : start + config.batch_size]
        chunks = [
            batch[i : i + config.micro_batch_size]
            for i in range(0, len(batch), config.micro_batch_size)
        ]
        chunk_labels = []
        for chunk in chunks:
            labels = tokenizer(
                [example.target for example in chunk],
                max_length=config.max_target_length,
                truncation=True,
                padding=True,
                return_tensors="pt",
            ).input_ids.to(device)
            labels[labels == tokenizer.pad_token_id] = -100
            chunk_labels.append(labels)
        batch_tokens = sum(int((labels != -100).sum()) for labels in chunk_labels)
        optimizer.zero_grad(set_to_none=True)
        batch_loss = 0.0
        for chunk, labels in zip(chunks, chunk_labels, strict=True):
            with torch.autocast(device.type, enabled=use_amp):
                loss = model(**encode_sources(chunk), labels=labels, use_cache=False).loss
            # Token-weighted so the summed gradient equals the full-batch mean-token gradient.
            weighted = loss * int((labels != -100).sum()) / batch_tokens
            scaler.scale(weighted).backward()
            batch_loss += weighted.item()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        losses.append(batch_loss)
        if step % report_every == 0 or step == steps_per_epoch:
            elapsed = time.time() - started
            print(
                f"  step {step}/{steps_per_epoch} ({100 * step / steps_per_epoch:.0f}%)"
                f" | loss {np.mean(losses[-report_every:]):.4f} | elapsed {clock(elapsed)}"
                f" | epoch time left ~{clock(elapsed / step * (steps_per_epoch - step))}",
                flush=True,
            )
    train_seconds = time.time() - started
    if device.type == "cuda":
        peak_memory_gb = round(torch.cuda.max_memory_allocated() / 1024**3, 2)
        print(f"  peak GPU memory so far: {peak_memory_gb} GB", flush=True)
    _, summary = evaluate(
        retrieved_examples, generate(retrieved_examples, f"epoch {epoch + 1} dev")
    )
    history.append(
        {
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "train_seconds": round(train_seconds, 1),
            "epoch_seconds_with_dev": round(time.time() - started, 1),
            **{f"dev_retrieved_{key}": value for key, value in summary.items() if key != "records"},
        }
    )
    improved = summary["execution_accuracy"] > best_accuracy
    if improved:
        best_accuracy = summary["execution_accuracy"]
        model.save_pretrained(model_dir / "best")
        tokenizer.save_pretrained(model_dir / "best")
    (model_dir / "history.json").write_text(
        json.dumps({"environment": environment, "history": history}, indent=2)
    )
    per_epoch = (time.time() - run_started) / (epoch + 1)
    print(
        f"Epoch {epoch + 1}/{epochs} done: {history[-1]}"
        f" | {'new best, checkpoint saved' if improved else 'not better than best'}"
        f" | total elapsed {clock(time.time() - run_started)}"
        f" | ~{clock(per_epoch * (epochs - epoch - 1))} left for {epochs - epoch - 1} epoch(s)",
        flush=True,
    )
display(pd.DataFrame(history))

## 4. Selected checkpoint: top beam and grounded selection

All 4 beam candidates are decoded; `top_beam` takes the first, `grounded` takes the first that executes
and uses only numbers in the facts (Notebook 14). Retrieved inputs and the gold-facts oracle are both
reported.

In [ ]:
@torch.no_grad()
def generate_candidates(examples, label: str) -> list[list[str]]:
    """All num_beams final beam candidates per input, in beam-score order."""
    model.eval()
    candidates, started = [], time.time()
    for start in range(0, len(examples), config.eval_batch_size):
        batch = examples[start : start + config.eval_batch_size]
        with torch.autocast(device.type, enabled=use_amp):
            output = model.generate(
                **encode_sources(batch),
                max_new_tokens=config.max_target_length,
                num_beams=num_beams,
                num_return_sequences=num_beams,
            )
        texts = tokenizer.batch_decode(
            output, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        candidates.extend(texts[i : i + num_beams] for i in range(0, len(texts), num_beams))
    print(f"  {label}: decoded {len(candidates)} ({clock(time.time() - started)})", flush=True)
    return candidates


model = AutoModelForSeq2SeqLM.from_pretrained(model_dir / "best").to(device)
results, all_scores = {}, {}
for condition, examples in [
    ("retrieved", retrieved_examples),
    ("gold_facts_oracle", oracle_examples),
]:
    candidates = generate_candidates(examples, f"selected checkpoint, {condition}")
    for selection in ("top_beam", "grounded"):
        chosen = [
            options[
                0
                if selection == "top_beam"
                else select_grounded_candidate(options, example.fact_texts, record["table"])
            ]
            for record, example, options in zip(dev, examples, candidates, strict=True)
        ]
        scores, summary = evaluate(examples, chosen)
        name = f"{condition}_{selection}"
        results[name], all_scores[name] = summary, scores
        path = predictions_dir / f"{config.experiment_name}{suffix}_dev_{name}_predictions.jsonl"
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("w", encoding="utf-8") as handle:
            for example, score in zip(examples, scores, strict=True):
                row = {**vars(score), "fact_keys": example.fact_keys, "commit": commit}
                handle.write(json.dumps(row) + "\n")
        print(f"  wrote {path.relative_to(PROJECT_ROOT)}", flush=True)
display(pd.DataFrame(results).T)

## 5. Paired comparison with the Notebook 11 checkpoint (retrieved, top beam, dev 0.5923)

Fixed means the Notebook 11 checkpoint was wrong and this one right; broken is the reverse. The current
pipeline (Notebook 11 + grounded selection) had 547 correct; compare the `retrieved_grounded` row above
with that.

In [ ]:
from scipy.stats import binomtest

BASELINE = predictions_dir / "generator-t5-base-dev-v1_dev_retrieved_predictions.jsonl"
if not BASELINE.exists() or SMOKE_TEST:
    print("Baseline prediction file not available here; paired comparison skipped.")
else:
    baseline = {
        json.loads(line)["record_id"]: json.loads(line)["execution_correct"]
        for line in BASELINE.read_text().splitlines()
    }
    rows = {}
    for name in ("retrieved_top_beam", "retrieved_grounded"):
        new = {score.record_id: score.execution_correct for score in all_scores[name]}
        fixed = sum(new[i] and not baseline[i] for i in baseline)
        broken = sum(baseline[i] and not new[i] for i in baseline)
        p_value = binomtest(fixed, fixed + broken).pvalue if fixed + broken else 1.0
        rows[name] = {"fixed": fixed, "broken": broken, "net": fixed - broken, "p_value": p_value}
    display(pd.DataFrame(rows).T)